In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from preprocessing import get_preprocessing_attributes

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
data = pd.read_csv("./dataset/data.csv")
data.drop_duplicates(subset=['title'], inplace=True)

In [5]:
data_train, data_val = train_test_split(data, test_size=0.2, random_state=42, stratify=data['label'])

In [6]:
data_train_preprocessed = get_preprocessing_attributes(data_train)
data_val_preprocessed = get_preprocessing_attributes(data_val)

In [11]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments

MODEL = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def tokenize(batch):
    return tokenizer(
        batch['preprocessed_text'],
        truncation=True,
        max_length=256,
    )
    
data_train_hugging = Dataset.from_pandas(data_train_preprocessed.reset_index(drop=True))
data_val_hugging = Dataset.from_pandas(data_val_preprocessed.reset_index(drop=True))

data_train_tokenized = data_train_hugging.map(tokenize, batched=True)
data_val_tokenized = data_val_hugging.map(tokenize, batched=True)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

Map:   0%|          | 0/28866 [00:00<?, ? examples/s]

Map:   0%|          | 0/7217 [00:00<?, ? examples/s]

In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall,
    }

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./deberta_fakenews",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    disable_tqdm=False,
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    num_train_epochs=3,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data_train_tokenized,
    eval_dataset=data_val_tokenized,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
c:\Users\agcor\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./deberta_finetuned_model")
tokenizer.save_pretrained("./deberta_finetuned_model")

In [13]:
import joblib

tfidf_pipeline = joblib.load("nlp_model.pkl")

scores = tfidf_pipeline.decision_function(data_val_preprocessed['preprocessed_text'])

if scores.ndim == 1:
    scores = np.vstack([-scores, scores]).T

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

tfidf_probs = softmax(scores)
tfidf_probs

array([[0.08281123, 0.91718877],
       [0.09544534, 0.90455466],
       [0.14016249, 0.85983751],
       ...,
       [0.12670586, 0.87329414],
       [0.90628764, 0.09371236],
       [0.11608088, 0.88391912]], shape=(7217, 2))

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="./tmp_predict",
    per_device_eval_batch_size=32,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=DataCollatorWithPadding(tokenizer)
)

test_ds = Dataset.from_dict({"preprocessed_text": data_val_preprocessed['preprocessed_text']}).map(tokenize, batched=True)
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

preds = trainer.predict(test_ds)
deberta_probs = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()

Map:   0%|          | 0/7217 [00:00<?, ? examples/s]

c:\Users\agcor\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: stack expects each tensor to be equal size, but got [82] at entry 0 and [145] at entry 1

In [ ]:
model_path = "./deberta_finetuned_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
def predict_deberta(texts, batch_size=32, max_length=256):
    all_probs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(
                batch, truncation=True, padding=True,
                max_length=max_length, return_tensors="pt"
            ).to(device)
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

deberta_probs = predict_deberta(data_val_preprocessed['preprocessed_text'].tolist())
deberta_preds = np.argmax(deberta_probs, axis=1)

In [ ]:
w_tfidf, w_deberta = 0.3, 0.7
final_probs = w_tfidf * tfidf_probs + w_deberta * deberta_probs
final_preds = np.argmax(final_probs, axis=1)